# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and perform basic analysis on the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described and structured by a Croissant schema (JSON-LD) accessible at a public URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset's Croissant package
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and all `@id` identifiers.
 
Below, we enumerate the record sets defined in the dataset. For each, we display their fields with field names and `@id`s.

> All RecordSets, Fields, and Columns are referenced by their `@id`. This ensures precise mapping for data extraction and reproducibility.

In [ ]:
# Helper to get record set info from Croissant metadata
def list_record_sets(ds):
    recsets = []
    if hasattr(ds.metadata, 'record_sets'):
        recsets = ds.metadata.record_sets
    elif hasattr(ds.metadata, 'recordSet'):
        recsets = ds.metadata.recordSet
    if not recsets:
        print("No record sets found in the metadata.")
        return []
    
    for rs in recsets:
        print(f"- RecordSet Name: {getattr(rs, 'name', 'N/A')} \n  @id: {getattr(rs, '@id', getattr(rs, 'id', 'N/A'))}")
        if hasattr(rs, 'fields'):
            fields = rs.fields
        elif hasattr(rs, 'field'):
            fields = rs.field
        else:
            fields = []
        print("  Fields:")
        for f in fields:
            print(f"    - {getattr(f, 'name', 'N/A')} (@id: {getattr(f, '@id', getattr(f, 'id', 'N/A'))})")
    return recsets

record_sets = list_record_sets(dataset)
if record_sets:
    # Print a sample of records from the first record set (if available)
    record_set_id = getattr(record_sets[0], '@id', getattr(record_sets[0], 'id', None))
    print(f"\nSample records from record set: {record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("Dataset contains no record sets explicitly defined.")

## 3. Data Extraction

We will extract data from each available record set into Pandas DataFrames using their `@id`. Fields (columns) are mapped by their respective `@id` as determined above.

If the dataset uses a single default record set, we will use its ID.

In [ ]:
# If there are record sets, extract by @id; otherwise try loading default records
dataframes = {}
record_set_ids = []
if record_sets:
    record_set_ids = [getattr(rs, '@id', getattr(rs, 'id', None)) for rs in record_sets]
else:
    print("No explicit record sets detected. Attempting to load default records...")
    try:
        # Try default loading
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['default'] = df
        print(df.head())
    except Exception as e:
        print(f"Error loading records: {e}")
    record_set_ids = ['default']

for record_set_id in record_set_ids:
    if record_set_id == 'default':
        continue
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)

Here we perform some basic data processing steps:

- Filter records using a numeric field and a threshold
- Normalize a numeric field
- Group records by a categorical field

All fields are referenced strictly by their `@id`. For purposes of this notebook, you may adjust field names/IDs according to the outputs above.

_Note: The appropriate `@id`s for fields must be used in the cells below. If you're not sure, print the columns above and update accordingly._

In [ ]:
# Choose the record set and field @id for analysis (update as needed)
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key].copy()
    print(f"Using DataFrame for record set: {df_key}")
    # Print available columns to select
    print("Available columns:", df.columns.tolist())
    
    # Example: use field IDs (replace as appropriate for the actual schema IDs)
    # Choose first numeric-like column (heuristically)
    numeric_field_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64]]
    if not numeric_field_candidates:
        # Try to convert
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_field_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64]]
    
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field for filtering: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another column that is categorical-like
        group_field_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No DataFrame loaded. Modify extraction above as needed.")

## 5. Visualization

Visualize the distribution of a numeric field and show relationships to a categorical field if available.

> Use `matplotlib` and `seaborn` for simple histograms, boxplots, or bar plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Continue using filtered_df, numeric_field and group_field if available
if 'filtered_df' in locals() and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Insufficient data for plotting. Please check data extraction/cell sequence.')

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load dataset metadata and records from a FAIR² Croissant dataset using `mlcroissant`
- List available record sets and fields by their `@id`
- Extract data and perform basic exploratory data analysis and visualization

Further analysis can use detailed field IDs and domain knowledge for downstream machine learning or scientific applications.